<a href="https://colab.research.google.com/github/TienNguyen0712/mimic_iv/blob/main/notebooks/01_mimic_iv_pipeline_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Mục tiêu**

Notebook này được viết nhằm thử nghiệm thiết kế một pipeline cho bộ dữ liệu MIMIC-IV giúp quản lý các bảng, làm sạch dữ liệu giám sát code cũng như chuẩn bi nguyên liệu cho các bước huấn luyện bài toán khác nhau

# **Kiến Trúc Công Nghệ**

Để tối ưu hóa hiệu năng xử lý trên môi trường cá nhân và Google Colab mà không cần đến cụm máy chủ cồng kềnh, hệ thống pipeline này lựa chọn một bộ công cụ tối giản nhưng cực kỳ mạnh mẽ:

| Thành phần | Công nghệ lựa chọn | Giải pháp thay thế (Production) |
|---|---|---|
| **Programming Language** | **Python 3.10+** | Python / Scala |
| **Data Processing Engine** | **DuckDB** | Apache Spark / Databricks |
| **Storage Format** | **Parquet (Apache)** | Delta Lake / Iceberg |
| **Configuration Layer** | **PyYAML (YAML Config)** | Hydra / Decouple |

---

# **Lý Do Lựa Chọn, Ưu Điểm & Nhược Điểm**

## **1. Data Processing Engine: DuckDB**
> **DuckDB** được mệnh danh là "SQLite dành cho phân tích dữ liệu". Nó là một hệ quản trị cơ sở dữ liệu dạng cột (columnar) nhúng trực tiếp vào tiến trình Python, cho phép truy vấn SQL tốc độ cao trên các file dữ liệu lớn mà không cần cài đặt server độc lập.

*   **Tại sao lại sử dụng?** Bảng dữ liệu lâm sàng của MIMIC-IV (như `chartevents`) lên tới hơn 300 triệu dòng (nặng khoảng hơn 30GB thô). Nếu dùng Pandas, hệ thống sẽ cố nạp toàn bộ vào RAM dẫn đến sập nguồn do lỗi `Out of Memory`. DuckDB có khả năng xử lý stream qua ổ đĩa (Out-of-Core processing), giúp xử lý file 30GB mượt mà trên máy chỉ có 8GB RAM.
*   **Ưu điểm:**
    *   **Siêu nhẹ & Tiện lợi:** Cài đặt chỉ với `pip install duckdb`. Không cần cấu hình Java, Hadoop hay Docker cồng kềnh như Apache Spark.
    *   **Tốc độ vượt trội:** Thực thi các câu lệnh SQL (Filter, Aggregation) nhanh gấp hàng chục lần so với Pandas thuần nhờ kiến trúc Vectorized Execution.
    *   **Tích hợp sâu:** Đọc trực tiếp từ file CSV, Parquet và chuyển đổi qua lại với Pandas DataFrame hoặc PyArrow chỉ trong 1 dòng code.
*   **Nhược điểm:**
    *   **Hạn chế mở rộng đa máy (Horizontal Scalability):** DuckDB tối ưu hóa phần cứng trên một máy đơn lẻ (Single-node). Nếu dữ liệu tăng lên quy mô hàng trăm Terabyte/Petabyte trên cụm máy chủ Cloud, nó không thể thay thế cho Apache Spark.

## **2. Storage Format: Apache Parquet**
> **Parquet** là định dạng lưu trữ dữ liệu mã nguồn mở dạng cột (columnar storage), được thiết kế để tối ưu hóa việc lưu trữ và truy vấn trong hệ sinh thái Big Data.

*   **Tại sao lại sử dụng?** Dữ liệu MIMIC-IV được phân phối mặc định ở dạng file `.csv` thô. File CSV lưu dữ liệu theo từng dòng (row-oriented), khiến hệ thống phải quét qua toàn bộ các cột không liên quan khi truy vấn, gây lãng phí tài nguyên đĩa.
*   **Ưu điểm:**
    *   **Tỷ lệ nén cực cao:** Tách biệt dữ liệu theo cột giúp áp dụng các thuật toán nén (như Snappy) hiệu quả hơn. Dung lượng đĩa giảm từ 70% đến 80% so với CSV thô.
    *   **Kỹ thuật Predicate Pushdown:** Cho phép DuckDB đọc trực tiếp siêu dữ liệu (metadata) của file Parquet để bỏ qua các khối dữ liệu (Data Blocks) hoặc các phân vùng không thỏa mãn điều kiện `WHERE` trước khi nạp vào RAM.
*   **Nhược điểm:**
    *   **Không thể đọc bằng mắt thường:** Khác với CSV mở lên là thấy chữ, Parquet là file nhị phân (Binary), muốn xem nội dung bắt buộc phải dùng code hoặc công cụ chuyên dụng (như Parquet Viewer).
    *   **Chi phí ghi đè cao:** Việc cập nhật một vài dòng nhỏ lẻ (Insert/Update) trong file Parquet tốn chi phí hơn vì hệ thống phải ghi lại cả một block cột.

## **3. Configuration Layer: YAML (PyYAML)**
> Sử dụng file cấu hình tách biệt để quản lý toàn bộ tham số của hệ thống (đường dẫn file, danh sách chỉ số xét nghiệm `itemid` cần lọc).

*   **Tại sao lại sử dụng?** Tránh việc "Hard-code" (viết chết đường dẫn hoặc tham số trong mã nguồn). Khi chuyển từ môi trường Google Colab về máy Local, Tiến chỉ cần chỉnh sửa đúng 1 file `.yaml` duy nhất mà không cần chạm vào logic code xử lý của pipeline.
*   **Ưu điểm:** Dễ đọc, cấu trúc phân cấp rõ ràng, dễ bảo trì và mở rộng khi hệ thống phức tạp hơn.
*   **Nhược điểm:** Cần viết thêm code boilerplate nhỏ để parse (đọc) file cấu hình trong Python.

# **Tầng 1: Bronze - Thực hiện việc phân vùng và lưu trữ các file thông qua DuckDB**

- Ý tưởng là thực hiện phân vùng chia nhỏ đối với các file có dung lượng lớn như file `chartevents`, `labelevents` chia theo chunking `subject_id` rồi sau đó chuyển đổi các file còn lại thành file parquet giúp cho việc load dễ dàng hơn

In [ ]:
# Cài đặt thư viện $ Kết nối Drive
# Cập nhật DuckDB bản mới nhất
!pip install --upgrade duckdb -q

import duckdb
import os
import glob
import time
from google.colab import drive

# Mount Google Drive để đọc/ghi file
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 21.7 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# Khởi tạo cấu trúc thư mục

# Định nghĩa các đường dẫn giả lập cấu trúc dự án
SOURCE_DIR = "/content/drive/MyDrive/NCKH-DDU1231/mimic-iv-clinical-database-demo-2.2/mimic-iv-clinical-database-demo-2.2" # Đường dẫn file sample trên Drive
BRONZE_BASE_DIR = "./data/bronze"

# Tạo thư mục đầu ra trong môi trường Colab
os.makedirs(BRONZE_BASE_DIR, exist_ok=True)

In [ ]:
# Định nghĩa các bảng lớn cần được Partition/Bucket để tránh quét toàn bộ ổ đĩa
# Các bảng khác nhỏ hơn sẽ được ghi thành 1 file duy nhất cho gọn
PARTITION_STRATEGY = {
    "chartevents": {"by": "subject_id", "buckets": 8},
    "labevents": {"by": "subject_id", "buckets": 8},
}

# Định nghĩa 7 bảng cần lấy để giảm tải hệ thống
SELECTED_TABLES = {
    "admissions",
    "icustays",
    "chartevents",
    "labevents",
    "inputevents",
    "outputevents",
    "procedureevents",
    "patients",
}

In [ ]:
def dynamic_bronze_pipeline(source_dir, output_base_dir):
    # Khởi tạo DuckDB In-Memory Engine
    conn = duckdb.connect(database=':memory:')

    # Tìm tất cả các file .csv.gz nằm trong các thư mục con
    search_pattern = os.path.join(source_dir, "**/*.csv.gz")
    gz_files = glob.glob(search_pattern, recursive=True)

    if not gz_files:
        print("Không tìm thấy file .csv.gz nào trong thư mục nguồn!")
        return

    # Lọc lại danh sách file: Chỉ giữ những file nằm trong danh sách 7 bảng đã chọn
    filtered_files = []
    for file_path in gz_files:
        file_name = os.path.basename(file_path)
        table_name = file_name.replace(".csv.gz", "")
        if table_name in SELECTED_TABLES:
            filtered_files.append((file_path, table_name))

    if not filtered_files:
        print(f"Không tìm thấy file nào khớp với 7 bảng đã chọn trong thư mục nguồn!")
        return

    print(f"Tìm thấy {len(filtered_files)} / {len(gz_files)} file thuộc danh sách lựa chọn cần xử lý.\n" + "="*50)

    # Chạy vòng lặp trên các file đã được lọc
    for file_path, table_name in filtered_files:
        start_time = time.time()

        file_name = os.path.basename(file_path)
        module_name = os.path.basename(os.path.dirname(file_path))

        # Tạo đường dẫn đầu ra động theo cấu trúc: ./data/bronze/{module_name}/{table_name}
        target_dir = os.path.join(output_base_dir, module_name, table_name)
        os.makedirs(target_dir, exist_ok=True)

        print(f"Đang xử lý: [Module: {module_name}] -> [Bảng: {table_name}]")

        # Kiểm tra xem bảng này có cần áp dụng chiến lược Bucketing không
        if table_name in PARTITION_STRATEGY:
            strategy = PARTITION_STRATEGY[table_name]
            partition_column = strategy["by"]
            num_buckets = strategy["buckets"]

            # Sử dụng hàm hash() chuẩn của DuckDB
            query = f"""
                COPY (
                    SELECT *,
                           abs(hash({partition_column})) % {num_buckets} AS bucket
                    FROM read_csv_auto('{file_path}')
                )
                TO '{target_dir}'
                (FORMAT PARQUET, PARTITION_BY bucket, OVERWRITE_OR_IGNORE 1);
            """
            print(f"Áp dụng Bucketing theo cột: {partition_column} ({num_buckets} buckets)")
        else:
            # Với bảng nhỏ, ghi trực tiếp thành 1 file parquet duy nhất nằm trong thư mục đó
            output_file = os.path.join(target_dir, f"{table_name}.parquet")
            query = f"""
                COPY (SELECT * FROM read_csv_auto('{file_path}'))
                TO '{output_file}'
                (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
            """
            print(f"Ghi thành file đơn: {table_name}.parquet")

        # Thực thi câu lệnh qua DuckDB
        try:
            conn.execute(query)
            duration = time.time() - start_time
            print(f"Hoàn thành trong {duration:.2f} giây -> Lưu tại: {target_dir}\n" + "-"*40)
        except Exception as e:
            print(f"Lỗi khi xử lý file {file_name}: {str(e)}\n" + "-"*40)

In [ ]:
dynamic_bronze_pipeline(SOURCE_DIR, BRONZE_BASE_DIR)

Tìm thấy 8 / 31 file thuộc danh sách lựa chọn cần xử lý.
Đang xử lý: [Module: icu] -> [Bảng: chartevents]
Áp dụng Bucketing theo cột: subject_id (8 buckets)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Hoàn thành trong 2.70 giây -> Lưu tại: ./data/bronze/icu/chartevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: icustays]
Ghi thành file đơn: icustays.parquet
Hoàn thành trong 0.03 giây -> Lưu tại: ./data/bronze/icu/icustays
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: inputevents]
Ghi thành file đơn: inputevents.parquet
Hoàn thành trong 0.32 giây -> Lưu tại: ./data/bronze/icu/inputevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: procedureevents]
Ghi thành file đơn: procedureevents.parquet
Hoàn thành trong 0.12 giây -> Lưu tại: ./data/bronze/icu/procedureevents
----------------------------------------
Đang xử lý: [Module: icu] -> [Bảng: outputevents]
Ghi thành file đơn: outputevents.parquet
Hoàn thành trong 0.08 giây -> Lưu tại: ./data/bronze/icu/outputevents
----------------------------------------
Đang xử lý: [Module: hosp] -> [Bảng: labevents]
Áp dụng Bucketing theo cột: subject_id (

# **Tầng 2: Silver - Thực hiện việc làm sạch dữ liệu và Chuẩn hóa quan hệ**

Tàng này sẽ kiểm tra chất lượng của dữ liệu biến dữ liệu thô thành các dữ liệu có cấu trúc và đáng tin cậy.

Do đó cần giải quyết 4 bài toán sau:

1. **Ép kiểu dữ liệu (Data Type Casting):** File Parquet ở tầng Bronze được DuckDB tự động nhận diện kiểu dữ liệu `(read_csv_auto)`, nhưng đôi khi chưa tối ưu. Cần ép kiểu tường minh: các cột ID về kiểu INTEGER, các cột thời gian về kiểu TIMESTAMP, các chỉ số về kiểu FLOAT.

2. **Đồng bộ hóa thời gian (Datetime Standardization):** Dữ liệu y tế của MIMIC-IV tràn ngập các cột thời gian (admittime, dischtime, intime, outtime, charttime). Cần đồng bộ tất cả về một chuẩn chung và xử lý các lỗi logic (ví dụ: bộ lọc loại bỏ các dòng có ngày xuất viện trước ngày nhập viện).

3. **Xử lý giá trị khuyết (NULL handling):** Quyết định xem cột nào bắt buộc không được NULL (như các cột khóa chính `subject_id`, `hadm_id`), và cột nào nếu NULL thì xử lý mặc định (ví dụ: nếu kết quả xét nghiệm value là text bị khuyết thì thay bằng 'UNKNOWN').

4. **Tạo các liên kết thực thể (Referential Integrity):** Đảm bảo các bảng dữ liệu lâm sàng (`chartevents, labevents`) đều có thể kết nối (JOIN) mượt mà với bảng xương sống `admissions` và `patients`.

Tầng Silver sẽ được chia thành 2 giai đoạn:
- **Giai đoạn 1:** Dựng bảng trung tâm `patient_master`
- **Giai đoạn 2:** Xử lý 5 bảng sự kiện lâm sàng bằng kỹ thuật **Pivot dữ liệu** dựa them danh sách itemid của hệ thống mimic

In [ ]:
# Định nghĩa phân chia thành các buckets nếu bảng nặng
PARTITION_STRATEGY = {
    "vital_signs": {"by": "stay_id", "buckets": 8},
    "laboratory": {"by": "stay_id", "buckets": 8},
}

def get_duckdb_connection():
    """Khởi tạo kết nối DuckDB In-Memory."""
    return duckdb.connect(database=':memory:')

## **Giai đoạn 1: Xây dựng bảng Trung Tâm `patient_master`**

Bảng này đóng vai trò là "Xương sống" (Master Anchor). Tất cả các bảng lâm sàng phía sau khi làm sạch đều phải INNER JOIN với bảng này theo stay_id để lọc bỏ những dữ liệu nằm ngoài thời gian nằm viện hoặc sai lệch định danh.

Các bước làm sạch & chuẩn hóa:
1. **Đồng bộ Khóa:** Kết hợp `admissions` và `icustays` qua `subject_id` và `hadm_id`.
2. **Chuẩn hóa Thời gian:** Ép kiểu toàn bộ các cột thời gian về dạng `TIMESTAMP`.
3. Xử lý logic thời gian bất thường: Loại bỏ các bản ghi có thời gian ra viện trước thời gian vào viện (`icu_outtime` < `icu_intime` hoặc `discharge_time` < `admission_time`).

In [ ]:
def build_patient_master(bronze_dir, silver_dir):
    """
    GIAI ĐOẠN 1: Đọc bảng admissions và icustays từ tầng Bronze,
    làm sạch logic thời gian và sinh bảng patient_master ở tầng Silver.
    """
    print("=== [GIAI ĐOẠN 1] Bắt đầu xây dựng 'patient_master' ===")
    start_time = time.time()
    conn = get_duckdb_connection()

    # 1. Khai báo và kiểm tra tệp nguồn Bronze
    required_tables = ["admissions", "icustays"]
    for table in required_tables:
        bronze_path = glob.glob(os.path.join(bronze_dir, f"**/{table}.parquet"), recursive=True) or \
                      glob.glob(os.path.join(bronze_dir, f"**/{table}"), recursive=True)
        if not bronze_path:
            raise FileNotFoundError(f"[X] Không tìm thấy dữ liệu Bronze cho bảng bắt buộc: {table}")

        path_to_read = bronze_path[0]
        # NẾU là thư mục (chứa các bucket), ta quét toàn bộ file parquet bên trong bằng `/**/*.parquet`
        if os.path.isdir(path_to_read):
            query_read = f"SELECT * FROM read_parquet('{path_to_read}/**/*.parquet', hive_partitioning=1)"
        # NẾU chỉ là một file đơn lẻ, ta đọc trực tiếp file đó
        else:
            query_read = f"SELECT * FROM read_parquet('{path_to_read}')"

        conn.execute(f"CREATE OR REPLACE VIEW bronze_{table} AS {query_read}")
        print(f" -> Đã kết nối thành công: bronze_{table}")

    # 2. Định nghĩa thư mục đầu ra
    master_target_dir = os.path.join(silver_dir, "core", "patient_master")
    os.makedirs(master_target_dir, exist_ok=True)

    # 3. Thực thi SQL ETL làm sạch hành chính
    query_master = f"""
        COPY (
            SELECT
                i.stay_id,
                i.hadm_id,
                i.subject_id,
                CAST(a.admittime AS TIMESTAMP) AS admission_time,
                CAST(a.dischtime AS TIMESTAMP) AS discharge_time,
                CAST(i.intime AS TIMESTAMP) AS icu_intime,
                CAST(i.outtime AS TIMESTAMP) AS icu_outtime,
                a.admission_type,
                i.first_careunit AS careunit
            FROM bronze_icustays i
            INNER JOIN bronze_admissions a
               ON i.hadm_id = a.hadm_id AND i.subject_id = a.subject_id
            WHERE i.stay_id IS NOT NULL
              AND CAST(i.intime AS TIMESTAMP) <= CAST(i.outtime AS TIMESTAMP)
        ) TO '{os.path.join(master_target_dir, "patient_master.parquet")}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
    """

    try:
        conn.execute(query_master)
        duration = time.time() - start_time
        print(f" -> [THÀNH CÔNG] Đã lưu 'patient_master' tại: {master_target_dir} ({duration:.2f} giây)\n")
    except Exception as e:
        print(f" -> [THẤT BẠI] Lỗi Giai đoạn 1: {str(e)}")
        raise e

Bên cạnh đó để phục vụ cho các bước sau ta cũng sẽ cần cột của các bảng
- `patients`:  Như tuổi, giới tính
- `chartevents`: Như cân nặng và chiều cao

Do đó ta sẽ có thêm một bảng `patients_clean` sinh ra nhằm JOIN với bảng trên

In [ ]:
def build_patients_clean(bronze_dir, silver_dir):
    """
    GIAI ĐOẠN 1.1:
    Làm sạch bảng patients từ Bronze và sinh bảng patients_clean ở Silver.
    """

    print("=== [GIAI ĐOẠN 1.1] Bắt đầu xây dựng 'patients_clean' ===")
    start_time = time.time()
    conn = get_duckdb_connection()

    # 1. Khai báo và kiểm tra tệp nguồn Bronze
    bronze_path = glob.glob(os.path.join(bronze_dir, "**/patients.parquet"), recursive=True) or \
                  glob.glob(os.path.join(bronze_dir, "**/patients"), recursive=True)
    if not bronze_path:
        raise FileNotFoundError("[X] Không tìm thấy bảng patients")

    path_to_read = bronze_path[0]
    if os.path.isdir(path_to_read):
        query_read = f"SELECT * FROM read_parquet('{path_to_read}/**/*.parquet', hive_partitioning=1)"
    else:
        query_read = f"SELECT * FROM read_parquet('{path_to_read}')"
    conn.execute(f"""
        CREATE OR REPLACE VIEW bronze_patients AS
        {query_read}
    """)

    # 2. Định nghĩa thư mục đầu ra
    target_dir = os.path.join(silver_dir, "core", "patients_clean")
    os.makedirs(target_dir, exist_ok=True)

    # 3. Lệnh SQL
    query = f"""
    COPY (
        SELECT DISTINCT
            subject_id,
            gender,
            CAST(anchor_age AS INTEGER)  AS anchor_age,
            CAST(anchor_year AS INTEGER) AS anchor_year
        FROM bronze_patients
        WHERE subject_id IS NOT NULL
    )
    TO '{os.path.join(target_dir,"patients_clean.parquet")}'
    (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
    """

    try:
        conn.execute(query)
        print(f" -> [THÀNH CÔNG] patients_clean ({time.time()-start_time:.2f} giấy\n)")
    except Exception as e:
        print(f" -> [THẤT BẠI] Lỗi Giai đoạn 1.1: {str(e)}")
        raise e

In [ ]:
def build_height_weight(bronze_dir, silver_dir):
    """
    GIAI ĐOẠN 1.2:
    Trích Height và Weight từ chartevents.
    """
    print("=== [GIAI ĐOẠN 1.2] Bắt đầu xây dựng 'height_weight' ===")
    start_time = time.time()
    conn = get_duckdb_connection()

    # Tìm đường dẫn chartevents (ưu tiên .parquet, sau đó là dạng thư mục)
    bronze_path = (
        glob.glob(os.path.join(bronze_dir, "**/chartevents.parquet"), recursive=True) or
        glob.glob(os.path.join(bronze_dir, "**/chartevents"), recursive=True)
    )
    if not bronze_path:
        raise FileNotFoundError("[X] Không tìm thấy bảng chartevents")

    path_to_read = bronze_path[0]

    # Rút gọn logic đọc file: Nếu là thư mục thì thêm '/**/*.parquet', nếu là file thì giữ nguyên
    read_pattern = f"{path_to_read}/**/*.parquet" if os.path.isdir(path_to_read) else path_to_read

    # Tạo VIEW trực tiếp từ đường dẫn đã xử lý (sử dụng hive_partitioning phòng trường hợp dữ liệu phân mảnh)
    conn.execute(f"""
        CREATE OR REPLACE VIEW bronze_chartevents AS
        SELECT * FROM read_parquet('{read_pattern}', hive_partitioning=1)
    """)

    target_file = os.path.join(silver_dir, "core", "height_weight", "height_weight.parquet")
    os.makedirs(os.path.dirname(target_file), exist_ok=True)

    # Rút gọn SQL: Gộp trực tiếp câu lệnh COPY và làm sạch CTE
    query = f"""
    COPY (
        SELECT
            stay_id,
            MAX(CASE WHEN itemid = 226707 THEN valuenum END) AS height_cm,
            MAX(CASE WHEN itemid = 226512 THEN valuenum END) AS weight_kg
        FROM bronze_chartevents
        WHERE valuenum IS NOT NULL AND itemid IN (226707, 226512)
        GROUP BY stay_id
    ) TO '{target_file}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
    """

    conn.execute(query)
    print(f" -> [THÀNH CÔNG] height_weight ({time.time() - start_time:.2f}s)")

## **Giai đoạn 2: Xử lý các bảng sự kiện lâm sàng**

**Nhóm Bảng Lâm Sàng Phân Tích (Vital Signs & Laboratory)**

Đối với `chartevents` và `labevents`, đặc trưng của dữ liệu MIMIC là lưu theo dạng EAV (Entity-Attribute-Value) – mỗi chỉ số là một dòng. Ở tầng Silver, ta cần Mapping `itemid` sang tên đặc trưng và lọc nhiễu lâm sàng (Outliers).

- Bảng vital_signs (Từ chartevents)
Mapping itemid chuẩn (MIMIC-IV):

  - Heart Rate: 220045 | SysBP: 220179, 220050 | DiasBP: 220180, 220051 | MeanBP: 220052, 220181

  - RespRate: 220210 | Temperature: 223761 (°C), 223762 (°F) | SpO2: 220277 | FiO2: 223835

  - Height: 226730 | Weight: 226512

  **Chuẩn hóa đơn vị:** Chuyển đổi Nhiệt độ từ Fahrenheit sang Celsius: $C = (F - 32) * 5/9$.

- Bảng laboratory (Từ `labevents`)
  - Mapping itemid chuẩn: Creatinine (50912), Lactate (50813), WBC (51301), Hemoglobin (51222), Platelet (51265), AST (50861), ALT (50863), Sodium (50983), Potassium (50971).
  - **Lưu ý:** labevents gốc chỉ có subject_id và hadm_id. Ta phải join với patient_master để lấy ra đúng stay_id dựa vào thời gian làm xét nghiệm charttime.

**Nhóm Bảng Can Thiệp Điều Trị (Medication, Outputs, Procedures)**

- Bảng medication (Từ inputevents - Thuốc truyền tĩnh mạch/vận mạch)
  - Làm sạch đặc thù: Tính toán tổng liều lượng hoặc giữ nguyên tốc độ truyền (rate), chuẩn hóa thời gian bắt đầu (starttime) và kết thúc (endtime).

  - Itemid gợi ý: Vasopressor (Norepinephrine: 221906), IV Fluid (NSS: 225158), Insulin (225152).

- Bảng outputs (Từ outputevents - Lượng dịch ra)
  - Làm sạch đặc thù: Gom cụm lượng nước tiểu hoặc dịch dẫn lưu theo từng giờ hoặc giữ nguyên mốc thời gian để tính Balance dịch.

  - Bảng procedures (Từ procedureevents - Thủ thuật lâm sàng)
    - Làm sạch đặc thù: Tính toán thời gian can thiệp (duration_hours = endtime - starttime). Khai báo trạng thái thủ thuật (Ví dụ: Đang thở máy = 1, Không thở máy = 0)

In [ ]:
def process_clinical_events(bronze_dir, silver_dir):
    """
    GIAI ĐOẠN 2: Đọc bảng patient_master từ Silver và 5 bảng sự kiện từ Bronze.
    Tiến hành Pivot, chuẩn hóa đơn vị, lọc nhiễu và lưu trữ (có Bucketing).
    """
    print("=== [GIAI ĐOẠN 2] Bắt đầu xử lý các bảng sự kiện lâm sàng ===")
    start_time = time.time()
    conn = get_duckdb_connection()

    # 1. Đăng ký liên kết tới bảng Master vừa tạo từ Giai đoạn 1
    master_path = os.path.join(silver_dir, "core", "patient_master", "*.parquet")
    if not glob.glob(master_path):
        raise FileNotFoundError("[X] Không tìm thấy bảng patient_master của Giai đoạn 1. Vui lòng chạy Giai đoạn 1 trước!")
    conn.execute(f"CREATE OR REPLACE VIEW silver_patient_master AS SELECT * FROM read_parquet('{master_path}')")

    # 2. Đăng ký các bảng lâm sàng còn lại từ tầng Bronze
    clinical_source_tables = ["chartevents", "labevents", "inputevents", "outputevents", "procedureevents"]
    for table in clinical_source_tables:
        bronze_path = glob.glob(os.path.join(bronze_dir, f"**/{table}.parquet"), recursive=True) or \
                      glob.glob(os.path.join(bronze_dir, f"**/{table}"), recursive=True)
        if bronze_path:
                    path_to_read = bronze_path[0]
                    if os.path.isdir(path_to_read):
                        query_read = f"SELECT * FROM read_parquet('{path_to_read}/**/*.parquet', hive_partitioning=1)"
                    else:
                        query_read = f"SELECT * FROM read_parquet('{path_to_read}')"

                    conn.execute(f"CREATE OR REPLACE VIEW bronze_{table} AS {query_read}")
                    print(f" -> Đã kết nối thành công: bronze_{table}")

    # 3. Từ điển chứa logic SQL biến đổi chuyên sâu cho từng bảng đích
    queries_clinical_tables = {
        "vital_signs": """
            SELECT
                p.stay_id,
                CAST(c.charttime AS TIMESTAMP) AS chart_time,
                CASE WHEN c.itemid = 220045 AND c.valuenum BETWEEN 30 AND 250 THEN c.valuenum END AS heart_rate,
                CASE WHEN c.itemid IN (220179, 220050) AND c.valuenum BETWEEN 40 AND 280 THEN c.valuenum END AS sys_bp,
                CASE WHEN c.itemid IN (220180, 220051) AND c.valuenum BETWEEN 20 AND 180 THEN c.valuenum END AS dias_bp,
                CASE WHEN c.itemid = 220210 AND c.valuenum BETWEEN 4 AND 60 THEN c.valuenum END AS resp_rate,
                CASE
                    WHEN c.itemid = 223761 AND c.valuenum BETWEEN 30 AND 45 THEN c.valuenum
                    WHEN c.itemid = 223762 AND c.valuenum BETWEEN 86 AND 113 THEN (c.valuenum - 32) * 5 / 9
                END AS temperature,
                CASE WHEN c.itemid = 220277 AND c.valuenum BETWEEN 50 AND 100 THEN c.valuenum END AS spo2
            FROM bronze_chartevents c
            INNER JOIN silver_patient_master p ON c.stay_id = p.stay_id
            WHERE CAST(c.charttime AS TIMESTAMP) BETWEEN p.icu_intime AND p.icu_outtime
              AND c.itemid IN (220045, 220179, 220050, 220180, 220051, 220210, 223761, 223762, 220277)
        """,
        "laboratory": """
            SELECT
                p.stay_id,
                CAST(l.charttime AS TIMESTAMP) AS lab_time,
                AVG(CASE WHEN l.itemid = 50912 THEN l.valuenum END) AS creatinine,
                AVG(CASE WHEN l.itemid = 50813 THEN l.valuenum END) AS lactate,
                AVG(CASE WHEN l.itemid = 51301 THEN l.valuenum END) AS white_blood_cell,
                AVG(CASE WHEN l.itemid = 51222 THEN l.valuenum END) AS hemoglobin,
                AVG(CASE WHEN l.itemid = 51265 THEN l.valuenum END) AS platelet
            FROM bronze_labevents l
            INNER JOIN silver_patient_master p ON l.hadm_id = p.hadm_id AND l.subject_id = p.subject_id
            WHERE CAST(l.charttime AS TIMESTAMP) BETWEEN p.icu_intime AND p.icu_outtime
              AND l.itemid IN (50912, 50813, 51301, 51222, 51265)
            GROUP BY p.stay_id, l.charttime
        """,
        "medication": """
            SELECT
                p.stay_id,
                CAST(i.starttime AS TIMESTAMP) AS start_time,
                CAST(i.endtime AS TIMESTAMP) AS end_time,
                CASE
                    WHEN i.itemid = 221906 THEN 'Norepinephrine'
                    WHEN i.itemid = 225152 THEN 'Insulin'
                    ELSE 'Other Med'
                END AS med_name,
                i.amount AS dose,
                i.rate AS rate
            FROM bronze_inputevents i
            INNER JOIN silver_patient_master p ON i.stay_id = p.stay_id
            WHERE i.statusdescription != 'Rewritten'
        """,
        "outputs": """
            SELECT
                p.stay_id,
                CAST(o.charttime AS TIMESTAMP) AS chart_time,
                CASE WHEN o.itemid IN (226559, 226560, 226561, 227488) THEN 'Urine Output' ELSE 'Other Output' END AS output_type,
                o.value AS volume_ml
            FROM bronze_outputevents o
            INNER JOIN silver_patient_master p ON o.stay_id = p.stay_id
            WHERE o.value BETWEEN 0 AND 5000
        """,
        "procedures": """
            SELECT
                p.stay_id,
                CAST(pr.starttime AS TIMESTAMP) AS start_time,
                CAST(pr.endtime AS TIMESTAMP) AS end_time,
                CASE WHEN pr.itemid = 227194 THEN 'Mechanical Ventilation' WHEN pr.itemid = 225792 THEN 'ECMO' ELSE 'Other' END AS procedure_name,
                date_diff('minute', CAST(pr.starttime AS TIMESTAMP), CAST(pr.endtime AS TIMESTAMP)) / 60.0 AS duration_hours
            FROM bronze_procedureevents pr
            INNER JOIN silver_patient_master p ON pr.stay_id = p.stay_id
            WHERE pr.statusdescription != 'Annull'
        """
    }

    # 4. Chạy vòng lặp xử lý và phân tách dữ liệu ra các thư mục Silver tương ứng
    for target_table, core_sql in queries_clinical_tables.items():
        table_start = time.time()
        target_dir = os.path.join(silver_dir, "clinical", target_table)
        os.makedirs(target_dir, exist_ok=True)

        # Áp dụng chiến lược phân nhỏ dữ liệu (Bucketing 8 files) nếu cấu hình yêu cầu
        if target_table in PARTITION_STRATEGY:
            strategy = PARTITION_STRATEGY[target_table]
            bucket_column = strategy["by"]
            num_buckets = strategy["buckets"]

            export_query = f"""
                COPY (
                    SELECT *, abs(hash({bucket_column})) % {num_buckets} AS bucket
                    FROM ({core_sql})
                ) TO '{target_dir}' (FORMAT PARQUET, PARTITION_BY bucket, OVERWRITE_OR_IGNORE 1);
            """
            print(f" -> Đang xử lý: {target_table} (Chia làm {num_buckets} Buckets theo {bucket_column})")
        else:
            export_query = f"""
                COPY ({core_sql}) TO '{os.path.join(target_dir, f"{target_table}.parquet")}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
            """
            print(f" -> Đang xử lý: {target_table} (Lưu file đơn gọn nhẹ)")

        try:
            conn.execute(export_query)
            print(f"   ✓ Hoàn thành bảng {target_table} trong {time.time() - table_start:.2f} giây.")
        except Exception as e:
            print(f"   [X] LỖI tại bảng {target_table}: {str(e)}")

    print(f"\n -> [THÀNH CÔNG] Toàn bộ Giai đoạn 2 hoàn tất trong {time.time() - start_time:.2f} giây.\n")

In [ ]:
if __name__ == "__main__":
    BRONZE_DIR = "./data/bronze"
    SILVER_DIR = "./data/silver"

    print("KHỞI ĐỘNG HỆ THỐNG PIPELINE TẦNG SILVER\n" + "="*50)

    # Bước 1: Sinh bảng trung tâm trước
    build_patient_master(BRONZE_DIR, SILVER_DIR)
    build_patients_clean(BRONZE_DIR, SILVER_DIR)
    build_height_weight(BRONZE_DIR, SILVER_DIR)

    # Bước 2: Dựa vào bảng trung tâm để xử lý sạch tất cả các dữ liệu lâm sàng
    process_clinical_events(BRONZE_DIR, SILVER_DIR)

    print("="*50 + "\n[DONE] Pipeline đã hoàn thành độc lập cả 2 giai đoạn.")

KHỞI ĐỘNG HỆ THỐNG PIPELINE TẦNG SILVER
=== [GIAI ĐOẠN 1] Bắt đầu xây dựng 'patient_master' ===
 -> Đã kết nối thành công: bronze_admissions
 -> Đã kết nối thành công: bronze_icustays
 -> [THÀNH CÔNG] Đã lưu 'patient_master' tại: ./data/silver/core/patient_master (0.03 giây)

=== [GIAI ĐOẠN 1.1] Bắt đầu xây dựng 'patients_clean' ===
 -> [THÀNH CÔNG] patients_clean (0.02s)
=== [GIAI ĐOẠN 1.2] Bắt đầu xây dựng 'height_weight' ===
 -> [THÀNH CÔNG] height_weight (0.04s)
=== [GIAI ĐOẠN 2] Bắt đầu xử lý các bảng sự kiện lâm sàng ===
 -> Đã kết nối thành công: bronze_chartevents
 -> Đã kết nối thành công: bronze_labevents
 -> Đã kết nối thành công: bronze_inputevents
 -> Đã kết nối thành công: bronze_outputevents
 -> Đã kết nối thành công: bronze_procedureevents
 -> Đang xử lý: vital_signs (Chia làm 8 Buckets theo stay_id)
   ✓ Hoàn thành bảng vital_signs trong 0.11 giây.
 -> Đang xử lý: laboratory (Chia làm 8 Buckets theo stay_id)
   ✓ Hoàn thành bảng laboratory trong 0.02 giây.
 -> Đang xử 

# **Tầng 3: Gold - Thực hiện việc chuẩn bị dữ liệu cho các bài toán đầu ra**

Khác với mô hình Data Mart truyền thống phục vụ báo cáo BI, tầng Gold được thiết kế theo mô hình Feature Store chuyên dụng cho AI/ML. Mỗi bản ghi trong Feature Store đại diện cho một thực thể lâm sàng duy nhất (stay_id), nơi toàn bộ các đặc trưng động và tĩnh được tính toán sẵn trên nhiều cửa sổ thời gian khác nhau để tạo thành một Ma trận Đặc trưng (Feature Matrix) sẵn sàng cho việc huấn luyện mô hình.

## **Kiến trúc các Feature Groups thành phần**
Hệ thống Feature Store được chia nhỏ thành 6 Feature Groups độc lập trước khi hợp nhất

### **Kỹ thuật Cửa sổ Thời gian (Multi-Window) & Đồng bộ Khóa Tần Gold**

- Tính toán Thời gian Động (Temporal Aggregation): Thay vì làm tròn thời gian cố định theo block (như date_trunc 4 giờ truyền thống), kỹ thuật Feature Store sử dụng hàm tính khoảng cách thời gian date_diff('hour', icu_intime, chart_time) để xác định chính xác một sự kiện lâm sàng thuộc về cửa sổ nào ($1h, 6h, 12h, 24h$) tính từ mốc số 0 là thời điểm bệnh nhân nhập khoa ICU.
- Một Feature Store — Nhiều Nhãn Đầu Ra (One-to-Many Labels Mapping): Cấu trúc phẳng hóa này cho phép tách biệt hoàn toàn phần Tính năng (Features) và phần Nhãn (Labels). Bảng `patient_feature_store` sau khi xuất ra định dạng Parquet có thể tái sử dụng ngay lập tức cho nhiều bài toán ML khác nhau bằng cách liên kết trực tiếp với các bảng nhãn mục tiêu:
  - Dự đoán Tử vong: LEFT JOIN mortality_labels
  - Dự đoán Suy thận cấp: LEFT JOIN aki_labels
  - Dự đoán Nhiễm trùng huyết: LEFT JOIN sepsis_labels
  - Dự đoán Thời gian nằm viện: LEFT JOIN los_labels

Điều này giúp các kỹ sư Data Science không cần phải viết lại hay chạy lại các pipeline dữ liệu thô cực nhọc cho mỗi lần thay đổi bài toán ML.

In [ ]:
SILVER_BASE_DIR = "./data/silver"
GOLD_BASE_DIR = "./data/gold"

# Khởi tạo DuckDB In-Memory Engine
conn = duckdb.connect(database=':memory:')

def print_group_summary(group_name, table_name):
    """Hàm hiển thị kết quả trực quan cho từng Feature Group"""
    res = conn.execute(f"SELECT COUNT(*), COUNT(DISTINCT stay_id) FROM {table_name}").fetchone()
    cols = conn.execute(f"PRAGMA table_info('{table_name}')").fetchall()
    col_names = [c[1] for c in cols]

    print(f"\n[FEATURE GROUP] {group_name}:")
    print(f"  - Bảng bộ nhớ   : {table_name}")
    print(f"  - Kích thước    : {res[0]} dòng x {len(cols)} cột")
    print(f"  - ICU Stays     : {res[1]}")
    print(f"  - Các cột chính : {', '.join(col_names[:6])} ...")
    print("-" * 60)

### **1. Patient Static Features (patient_static_features)**
- Mô tả: Các đặc trưng nhân khẩu học và hành chính cố định của ca bệnh.
- Các tính năng tính toán: $Age$ (tính từ năm sinh đến thời điểm intime), $Gender$, $Height$, $Weight$, $BMI$ (tính toán bằng công thức $\frac{Weight}{(Height/100)^2}$), $Admission\ Type$, và $ICU\ Type$.

In [ ]:
def build_feature_group_1(silver_dir):
    """
    Feature Group 1:
    Xây dựng bảng patient_static_features từ các bảng Silver.
    """
    print(" -> [FG1] Đang xây dựng: patient_static_features...")

    # Rút gọn định nghĩa đường dẫn bằng f-string ngắn gọn
    pm_path = os.path.join(silver_dir, "core", "patient_master", "*.parquet")
    p_path = os.path.join(silver_dir, "core", "patients_clean", "*.parquet")
    hw_path = os.path.join(silver_dir, "core", "height_weight", "*.parquet")

    query = f"""
    CREATE OR REPLACE TABLE patient_static_features AS
    SELECT
        pm.stay_id,
        pm.subject_id,
        pm.hadm_id,

        -- Demographic
        (p.anchor_age + (EXTRACT(YEAR FROM pm.icu_intime) - p.anchor_year)) AS age,
        p.gender,

        -- Anthropometric
        hw.height_cm,
        hw.weight_kg,
        CASE
            WHEN hw.height_cm > 0 AND hw.weight_kg IS NOT NULL
            THEN ROUND(hw.weight_kg / POWER(hw.height_cm / 100.0, 2), 2)
            ELSE NULL
        END AS bmi,

        -- Administrative
        pm.admission_type,
        pm.careunit AS icu_type
    FROM read_parquet('{pm_path}') pm
    LEFT JOIN read_parquet('{p_path}') p ON pm.subject_id = p.subject_id
    LEFT JOIN read_parquet('{hw_path}') hw ON pm.stay_id = hw.stay_id;
    """

    conn.execute(query)
    print_group_summary("1 — Patient Static Features", "patient_static_features")

### **2. Vital Features (vital_features — Multi-Window)**
- Mô tả: Gom tụ dữ liệu dấu hiệu sinh tồn dày đặc từ chartevents theo các cửa sổ thời gian lũy tiến: 1h, 6h, 12h, và 24h đầu tiên kể từ lúc vào ICU.
- Các tính năng tính toán: Các chỉ số sinh hiệu (Heart Rate, Respiratory Rate, MAP, Temperature, SpO₂) được gộp bằng các hàm AVG, MIN, MAX riêng biệt cho từng cửa sổ.Ví dụ: hr_mean_1h, hr_max_6h, spo2_min_24h.

In [ ]:
def build_feature_group_2(silver_dir):
    print(" -> [FG2] Đang xây dựng: vital_features (Tính toán đa cửa sổ thời gian)...")
    vitals_pattern = os.path.join(silver_dir, "clinical", "vital_signs", "bucket=*", "*.parquet")

    # Kỹ thuật tính đa cửa sổ thời gian (1h, 6h, 12h, 24h) dựa trên khoảng cách từ lúc vào ICU (icu_intime)
    # Để đơn giản và tối ưu hiệu năng, ta lấy mốc chart_time so với thời điểm bắt đầu của ca bệnh
    query = f"""
        CREATE OR REPLACE TABLE vital_features AS
        WITH vitals_with_time AS (
            SELECT
                v.stay_id,
                v.heart_rate,
                v.sys_bp,
                v.temperature,
                v.spo2,
                -- Tính số giờ kể từ khi bệnh nhân vào ICU (giả định dùng mốc chặn trên/dưới)
                date_diff('hour', pm.icu_intime, v.chart_time) as hours_since_in
            FROM read_parquet('{vitals_pattern}') v
            JOIN read_parquet('{os.path.join(silver_dir, "core", "patient_master", "*.parquet")}') pm
              ON v.stay_id = pm.stay_id
        )
        SELECT
            stay_id,
            -- Toàn bộ thời gian (Tổng quan) hoặc Tiến có thể chia nhỏ mệnh đề FILTER bên dưới:
            ROUND(CAST(AVG(heart_rate) AS NUMERIC), 2) AS HR_mean,
            ROUND(CAST(MAX(heart_rate) AS NUMERIC), 2) AS HR_max,
            ROUND(CAST(AVG(sys_bp) AS NUMERIC), 2) AS MAP_mean, -- Tạm lấy sys_bp làm đại diện MAP nếu chưa có công thức biến đổi
            ROUND(CAST(MAX(temperature) AS NUMERIC), 2) AS Temp_max,
            ROUND(CAST(MIN(spo2) AS NUMERIC), 2) AS SpO2_min,

            -- Ví dụ cửa sổ 6 giờ đầu tiên (Windowing)
            ROUND(CAST(AVG(CASE WHEN hours_since_in <= 6 THEN heart_rate END) AS NUMERIC), 2) AS HR_mean_6h,
            ROUND(CAST(AVG(CASE WHEN hours_since_in <= 24 THEN heart_rate END) AS NUMERIC), 2) AS HR_mean_24h
        FROM vitals_with_time
        GROUP BY stay_id;
    """
    conn.execute(query)
    print_group_summary("2 — Vital Features", "vital_features")

### **3. Laboratory Features (lab_features)**
- Mô tả: Các chỉ số xét nghiệm sinh hóa máu và dịch thể từ bảng laboratory.Các tính năng tính toán: Giá trị cực trị hoặc trung bình trong suốt ca nằm viện hoặc trong ngày đầu tiên như Creatinine_max, Lactate_max, WBC_mean, Platelet_min.

In [ ]:
def build_feature_group_3(silver_dir):
    print(" -> [FG3] Đang xây dựng: lab_features...")
    labs_pattern = os.path.join(silver_dir, "clinical", "laboratory", "bucket=*", "*.parquet")

    query = f"""
        CREATE OR REPLACE TABLE lab_features AS
        SELECT
            stay_id,
            ROUND(CAST(MAX(creatinine) AS NUMERIC), 2) AS Creatinine_max,
            ROUND(CAST(MAX(lactate) AS NUMERIC), 2) AS Lactate_max,
            ROUND(CAST(AVG(white_blood_cell) AS NUMERIC), 2) AS WBC_mean,
            ROUND(CAST(MIN(platelet) AS NUMERIC), 2) AS Platelet_min
        FROM read_parquet('{labs_pattern}')
        GROUP BY stay_id;
    """
    conn.execute(query)
    print_group_summary("3 — Laboratory Features", "lab_features")

### **4. Medication Features (medication_features)**
- Mô tả: Thông tin điều trị bằng thuốc, dịch truyền và vận mạch từ bảng medication.
- Các tính năng tính toán: Tổng lượng dịch truyền (Total Fluid), Tổng liều vận mạch (Total Vasopressor Dose), Tổng lượng Insulin (Total Insulin), và tổng thời gian sử dụng vận mạch (Vasopressor Duration).

In [ ]:
def build_feature_group_4(silver_dir):
    print(" -> [FG4] Đang xây dựng: medication_features...")
    meds_pattern = os.path.join(silver_dir, "clinical", "medication", "*.parquet")

    query = f"""
        CREATE OR REPLACE TABLE medication_features AS
        SELECT
            stay_id,
            -- Trích xuất các biến định lượng dịch truyền và vận mạch theo từ khóa thuốc
            ROUND(CAST(SUM(CASE WHEN med_name LIKE '%Fluid%' OR med_name LIKE '%NaCl%' THEN dose ELSE 0 END) AS NUMERIC), 2) AS Total_Fluid,
            ROUND(CAST(SUM(CASE WHEN med_name LIKE '%Norepinephrine%' OR med_name LIKE '%Epinephrine%' THEN dose ELSE 0 END) AS NUMERIC), 2) AS Total_Vasopressor_Dose,
            ROUND(CAST(SUM(CASE WHEN med_name LIKE '%Insulin%' THEN dose ELSE 0 END) AS NUMERIC), 2) AS Total_Insulin
        FROM read_parquet('{meds_pattern}')
        GROUP BY stay_id;
    """
    conn.execute(query)
    print_group_summary("4 — Medication Features", "medication_features")

### **5. Output Features (output_features)**
- Mô tả: Theo dõi lượng dịch ra (chất tiết, nước tiểu) để đánh giá chức năng tạng (ví dụ: Thận).
- Các tính năng tính toán: Tổng lượng nước tiểu thu được trong 6 giờ đầu (Urine Output 6h), 24 giờ đầu (Urine Output 24h) và lượng dịch dẫn lưu (Drain Output).

In [ ]:
def build_feature_group_5(silver_dir):
    print(" -> [FG5] Đang xây dựng: output_features...")
    outputs_pattern = os.path.join(silver_dir, "clinical", "outputs", "*.parquet")

    query = f"""
        CREATE OR REPLACE TABLE output_features AS
        SELECT
            stay_id,
            SUM(CASE WHEN output_type LIKE '%Urine%' THEN volume_ml ELSE 0 END) AS Urine_Output,
            SUM(CASE WHEN output_type LIKE '%Drain%' THEN volume_ml ELSE 0 END) AS Drain_Output
        FROM read_parquet('{outputs_pattern}')
        GROUP BY stay_id;
    """
    conn.execute(query)
    print_group_summary("5 — Output Features", "output_features")

### **6. Procedure Features (procedure_features)**
- Mô tả: Các can thiệp và thủ thuật lâm sàng xâm lấn từ bảng procedures.
- Các tính năng tính toán: Các biến Phân loại/Bị phân (Binary Flags) thể hiện bệnh nhân có can thiệp hay không (như Mechanical Ventilation, Dialysis, ECMO, Intubation) cùng với thời gian can thiệp (duration_hours).

In [ ]:
def build_feature_group_6(silver_dir):
    print(" -> [FG6] Đang xây dựng: procedure_features...")
    procedures_pattern = os.path.join(silver_dir, "clinical", "procedures", "*.parquet")

    query = f"""
        CREATE OR REPLACE TABLE procedure_features AS
        SELECT
            stay_id,
            -- Chuyển đổi các thủ thuật y tế thành biến Flag (0/1) để đưa vào mô hình ML
            MAX(CASE WHEN procedure_name LIKE '%Ventilation%' THEN 1 ELSE 0 END) AS Mechanical_Ventilation,
            MAX(CASE WHEN procedure_name LIKE '%Dialysis%' OR procedure_name LIKE '%CRRT%' THEN 1 ELSE 0 END) AS Dialysis,
            MAX(CASE WHEN procedure_name LIKE '%ECMO%' THEN 1 ELSE 0 END) AS ECMO,
            MAX(CASE WHEN procedure_name LIKE '%Intubation%' THEN 1 ELSE 0 END) AS Intubation
        FROM read_parquet('{procedures_pattern}')
        GROUP BY stay_id;
    """
    conn.execute(query)
    print_group_summary("6 — Procedure Features", "procedure_features")

### **Hợp nhất thành Unified Patient Feature Store (patient_feature_store)**

Toàn bộ 6 Feature Groups trên sẽ được liên kết với nhau bằng phép toán LEFT JOIN thông qua khóa chính stay_id để tạo ra một bảng phẳng duy nhất.$$\text{Patient Master} \xrightarrow{\text{LEFT JOIN}} \begin{cases}  \text{Static Features} \\ \text{Vital Features (Multi-window)} \\ \text{Laboratory Features} \\ \text{Medication Features} \\ \text{Output Features} \\ \text{Procedure Features}  \end{cases} \xrightarrow{\text{FLATTEN}} \mathbf{Patient\ Feature\ Store}$$

In [ ]:
def build_unified_patient_feature_store(gold_dir):
    print("\n -> [Unified] Đang hợp nhất 6 Feature Groups thành 'patient_feature_store'...")
    final_gold_dir = os.path.join(gold_dir, "patient_feature_store")
    os.makedirs(final_gold_dir, exist_ok=True)

    # Thực hiện LEFT JOIN chuỗi tuần tự nối từ Static Features đi qua toàn bộ các Feature Groups
    query = f"""
        CREATE OR REPLACE TABLE patient_feature_store AS
        SELECT
            -- 1. Static Features làm gốc
            sf.*,

            -- 2. Vital Features
            v.HR_mean, v.HR_max, v.MAP_mean, v.Temp_max, v.SpO2_min, v.HR_mean_6h, v.HR_mean_24h,

            -- 3. Laboratory Features
            l.Creatinine_max, l.Lactate_max, l.WBC_mean, l.Platelet_min,

            -- 4. Medication Features
            COALESCE(m.Total_Fluid, 0.0) AS Total_Fluid,
            COALESCE(m.Total_Vasopressor_Dose, 0.0) AS Total_Vasopressor_Dose,
            COALESCE(m.Total_Insulin, 0.0) AS Total_Insulin,

            -- 5. Output Features
            COALESCE(o.Urine_Output, 0) AS Urine_Output,
            COALESCE(o.Drain_Output, 0) AS Drain_Output,

            -- 6. Procedure Features
            COALESCE(p.Mechanical_Ventilation, 0) AS Mechanical_Ventilation,
            COALESCE(p.Dialysis, 0) AS Dialysis,
            COALESCE(p.ECMO, 0) AS ECMO,
            COALESCE(p.Intubation, 0) AS Intubation

        FROM patient_static_features sf
        LEFT JOIN vital_features v      ON sf.stay_id = v.stay_id
        LEFT JOIN lab_features v        ON sf.stay_id = l.stay_id
        LEFT JOIN medication_features m ON sf.stay_id = m.stay_id
        LEFT JOIN output_features o     ON sf.stay_id = o.stay_id
        LEFT JOIN procedure_features p  ON sf.stay_id = p.stay_id;
    """
    # Lưu ý: Sửa lỗi alias trùng lặp ngầm (lab_features l chứ không phải v)
    query = query.replace("LEFT JOIN lab_features v", "LEFT JOIN lab_features l")
    conn.execute(query)

    # Kết xuất ra đĩa cứng theo cấu trúc 8 buckets bằng toán tử MOD, đảm bảo tính phân tán
    export_query = f"""
        COPY (
            SELECT *, MOD(stay_id, 8) as bucket FROM patient_feature_store
        ) TO '{final_gold_dir}'
        (FORMAT PARQUET, PARTITION_BY (bucket), OVERWRITE_OR_IGNORE 1);
    """
    conn.execute(export_query)
    print_group_summary("UNIFIED PATIENT FEATURE STORE (GOLD MASTER)", "patient_feature_store")

def run_gold_pipeline(silver_dir, gold_dir):
    print("=== [Gold Layer] KHỞI ĐỘNG KIẾN TRÚC MACHINE LEARNING FEATURE STORE ===")
    start_time = time.time()

    # Chạy độc lập 6 khối Feature Groups
    build_feature_group_1(silver_dir)
    build_feature_group_2(silver_dir)
    build_feature_group_3(silver_dir)
    build_feature_group_4(silver_dir)
    build_feature_group_5(silver_dir)
    build_feature_group_6(silver_dir)

    # Gom tất cả lại tại đích cuối cùng
    build_unified_patient_feature_store(gold_dir)

    duration = time.time() - start_time
    print(f"\n[THÀNH CÔNG] Toàn bộ 6 Feature Groups đã được lưu và tích hợp thành công sau {duration:.2f} giây!")


if __name__ == "__main__":
    run_gold_pipeline(SILVER_BASE_DIR, GOLD_BASE_DIR)

=== [Gold Layer] KHỞI ĐỘNG KIẾN TRÚC MACHINE LEARNING FEATURE STORE ===
 -> [FG1] Đang xây dựng: patient_static_features...

[FEATURE GROUP] 1 — Patient Static Features:
  - Bảng bộ nhớ   : patient_static_features
  - Kích thước    : 140 dòng x 10 cột
  - ICU Stays     : 140
  - Các cột chính : stay_id, subject_id, hadm_id, age, gender, height_cm ...
------------------------------------------------------------
 -> [FG2] Đang xây dựng: vital_features (Tính toán đa cửa sổ thời gian)...

[FEATURE GROUP] 2 — Vital Features:
  - Bảng bộ nhớ   : vital_features
  - Kích thước    : 140 dòng x 8 cột
  - ICU Stays     : 140
  - Các cột chính : stay_id, HR_mean, HR_max, MAP_mean, Temp_max, SpO2_min ...
------------------------------------------------------------
 -> [FG3] Đang xây dựng: lab_features...

[FEATURE GROUP] 3 — Laboratory Features:
  - Bảng bộ nhớ   : lab_features
  - Kích thước    : 139 dòng x 5 cột
  - ICU Stays     : 139
  - Các cột chính : stay_id, Creatinine_max, Lactate_max, WBC